# Dataset exploration outline

1. Load local validated data.
2. Inspect splits, labels, length and duplicates.
3. Keep CLINC out-of-scope examples separate.
4. Define validation and few-shot sampling using training data only.


# Environment check
Select the py3 kernel. This checks infrastructure, not research performance.

In [3]:
import sys, importlib.metadata as metadata, subprocess
import torch
import pandas as pd
import re

print(sys.executable)
for package in ['torch','setfit','sentence-transformers','transformers','datasets','scikit-learn']:
    print(package, metadata.version(package))
print('CUDA runtime:', torch.version.cuda)
assert torch.cuda.is_available(), 'CUDA unavailable'
print(torch.cuda.get_device_name(0))
x = torch.randn(64, 64, device='cuda', requires_grad=True)
(x @ x).sum().backward()
torch.cuda.synchronize()
print('GPU forward/backward passed')
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)

c:\APP\Anaconda\envs\py3\python.exe
torch 2.14.1+cu132
setfit 1.2.0
sentence-transformers 6.2.0
transformers 5.22.1
datasets 5.1.0
scikit-learn 1.9.1
CUDA runtime: 13.2
NVIDIA GeForce RTX 5060 Laptop GPU
GPU forward/backward passed


CompletedProcess(args=['c:\\APP\\Anaconda\\envs\\py3\\python.exe', '-m', 'pip', 'check'], returncode=0)

# Import dataset BANKING77

In [6]:
from datasets import load_dataset

banking = load_dataset("PolyAI/banking77")

print(banking)
print(banking["train"].features)
print(banking["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})
{'text': Value('string'), 'label': ClassLabel(names=['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire', 'card_acceptance', 'card_arrival', 'card_delivery_estimate', 'card_linking', 'card_not_working', 'card_payment_fee_charged', 'card_payment_not_recognised', 'card_payment_wrong_exchange_rate', 'card_swallowed', 'cash_withdrawal_charge', 'cash_withdrawal_not_recognised', 'change_pin', 'compromised_card', 'contactless_not_working', 'country_support', 'declined_card_payment', 'declined_cash_withdrawal', 'declined_transfer', 'direct_debit_payment_not_recognised', 'disposable_card_limits', 'edit

# Check CLINC150

In [7]:
clinc = load_dataset("DeepPavlov/clinc150","default")

print(clinc)
print(clinc["train"].features)
print(clinc["train"][0])


c:\APP\Anaconda\envs\py3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\l_len\.cache\huggingface\hub\datasets--DeepPavlov--clinc150. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 5500/5500 [00:00<00:00, 305622.24 examples/s]


DatasetDict({
    train: Dataset({
        features: ['utterance', 'label'],
        num_rows: 15200
    })
    validation: Dataset({
        features: ['utterance', 'label'],
        num_rows: 3100
    })
    test: Dataset({
        features: ['utterance', 'label'],
        num_rows: 5500
    })
})
{'utterance': Value('string'), 'label': Value('int64')}
{'utterance': 'can i make a reservation for redrobin', 'label': 0}


# Making dataset overview

In [8]:
train_df = banking["train"].to_pandas()

label_counts = train_df['label'].value_counts()

print(label_counts.describe())
print("Number of classes:", len(label_counts))
print("Minimum classes size:", label_counts.min())
print("Maximum classes size:", label_counts.max())


count     77.000000
mean     129.909091
std       32.942207
min       35.000000
25%      112.000000
50%      127.000000
75%      159.000000
max      187.000000
Name: count, dtype: float64
Number of classes: 77
Minimum classes size: 35
Maximum classes size: 187


# Experiment Setup Report

## Dataset Configuration

| Setting | BANKING77 | CLINC150 |
|---|---|---|
| Known intents | 77 | 150 |
| Training | K per class | K per class |
| Validation | 10–15% stratified split from original train | Official validation (known intents) |
| Test | Full official test | Official test (known intents) |
| OOS | N/A | Excluded |

## Procedure

- **BANKING77**: Hold out 10–15% of train as validation (fixed ratio), then sample K-shot only from the remaining pool.
- **CLINC150**: Keep official splits; exclude OOS from the main experiment.

## Critical Rules

- K-shot sampling from training pool only.
- Validation → hyperparameter selection.
- Test → final evaluation only (no leakage).

# 5、SetFit Pilot

## Pilot 001 — 最小可行实验

| 项目 | 设置 |
|---|---|
| Dataset | BANKING77 |
| Method | SetFit |
| Backbone | all-MiniLM-L6-v2 |
| Few-Shot Setting | 8 samples / class |
| Random Seed | 42 |
| Primary Metrics | Accuracy / Macro-F1 |


In [9]:
import random
import numpy as np
import torch
from datasets import load_dataset
from setfit import sample_dataset

SEED = 42

K = 8

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

dataset = load_dataset("PolyAI/banking77")

train_pool = dataset["train"].shuffle(seed=SEED)
fewshot_train = sample_dataset(train_pool, label_column="label",num_samples=K)

num_classes = dataset["train"].features["label"].num_classes

assert len(fewshot_train) == K * num_classes
print(f"Few-shot training dataset size: {len(fewshot_train)}")

print(f"Number of classes: {num_classes}")




Few-shot training dataset size: 616
Number of classes: 77


# Set up SetFit model

In [ ]:
from setfit import SetFitModel, TrainingArguments, Trainer
BACKBONE = "sentence-transformers/all-MiniLM-L6-v2"

model = SetFitModel.from_pretrained(BACKBONE)

args = TrainingArguments(
    output_dir="results/setfit_pilot",
    batch_size=16,
    num_epochs=1,
    num_iterations=5,
    body_learning_rate=2e-5,
    seed=SEED,
    report_to="none",
    save_strategy="no",
)

trainer = Trainer(model=model, args=args, train_dataset=fewshot_train,)

trainer.train()





Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6855.18it/s]
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Map: 100%|██████████| 616/616 [00:00<00:00, 1314.73 examples/s]
***** Running training *****
  Num unique pairs = 6160
  Batch size = 16
  Num epochs = 1


Step,Training Loss
1,0.073827
50,0.106612
100,0.083866
150,0.071585
200,0.069112
250,0.066047
300,0.060691
350,0.056444


In [16]:
from sklearn.metrics import accuracy_score, f1_score

predictions = model.predict(["Do I have any important meetings today?"])

print("Predicted class:", predictions)

Predicted class: tensor([12])


# Check function

In [ ]:
import re

def normalize_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def audit_overlap(split_a, split_b):
    a = split_a.copy()
    b = split_b.copy()
    a["text_key"] = a["text"].map(normalize_text)
    b["text_key"] = b["text"].map(normalize_text)
    overlap = a.merge(b, on="text_key", suffixes=("_a", "_b"))
    same_label = overlap[overlap["label_a"] == overlap["label_b"]]
    conflict = overlap[overlap["label_a"] != overlap["label_b"]]

    return {
        "overlap_count": len(overlap),
        "unique_text_count": len(overlap["text_key"].nunique()),
        "same_label_count": len(same_label),
        "conflict_count": len(conflict)
    }
